# Whisper Gradio Speech Recognition

Fresh Python3.11+ CPU kernel; pinned official Distil-Whisper weights and exact original LibriSpeech train-clean100 first5 clips, no API key. Requires FFmpeg on PATH for file decoding (install from official vendor/package manager if missing). Correct dataset config/split, remove unnecessary trust_remote_code, repair malformed with-demo line, modernize Gradio flagging_mode and provide local port fallback. All five full clips transcribed; API tests use actual audio files for both local routes, not physical microphone capture. Default127.0.0.1/shareFalse; opt in explicitly to public sharing. Regression cell closes its own server; rerun launch cell afterward to explore UI. Original source code/transcripts remain in appendices.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0", "transformers==4.57.6", "numpy==2.3.5", "datasets==3.6.0", "soundfile==0.13.1", "librosa==0.11.0", "gradio==6.3.0", "certifi==2026.10.23", "ipywidgets==8.1.7"], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import torch
import transformers
import soundfile as sf
import librosa
from pathlib import Path
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

## Example 0

In [ ]:
print("Pinned Transformers, datasets, soundfile, librosa and Gradio installed in the active kernel.")

## Example 1

In [ ]:
from datasets import load_dataset
DATA_REVISION = "71cacbfb7e2354c4226d01e70d77d5fca3d04ba1"
dataset = load_dataset("openslr/librispeech_asr", "clean", revision=DATA_REVISION, split="train.100", streaming=True, token=False, trust_remote_code=False)
print("Streaming original LibriSpeech clean/train.100 | revision:", DATA_REVISION)

## Example 2

In [ ]:
dataset_head = list(dataset.take(5))
example = dataset_head[0]
for record in dataset_head:
    print(record["id"], "|", len(record["audio"]["array"]), "samples |", record["audio"]["sampling_rate"], "Hz")
    print(record["text"])

## Example 3

In [ ]:
third_example = dataset_head[2]
print(third_example["id"])
print(third_example["text"])

## Example 4

In [ ]:
from IPython.display import Audio as IPythonAudio, display
display(IPythonAudio(example["audio"]["array"], rate=example["audio"]["sampling_rate"]))
print("Notebook audio player:", example["id"])

## Example 5

In [ ]:
from transformers import pipeline
MODEL_ID = "distil-whisper/distil-small.en"
MODEL_REVISION = "9e4a67ca4569c30be43a3fe7fba1621e504f0093"
asr = pipeline(task="automatic-speech-recognition", model=MODEL_ID, revision=MODEL_REVISION, token=False, device=-1, dtype=torch.float32)
asr.model.eval()
print("Loaded:", MODEL_ID, "| revision:", MODEL_REVISION)

## Example 6

In [ ]:
print(asr.feature_extractor.sampling_rate)

## Example 7

In [ ]:
print(example["audio"]["sampling_rate"])

## Example 8

In [ ]:
import gradio as gr
demo = gr.Blocks(analytics_enabled=False)
print("Gradio:", gr.__version__)
print("Local tabbed demo created.")

## Example 9

In [ ]:
def transcribe_speech(filepath):
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        return ""
    output = asr(filepath, generate_kwargs={"do_sample":False, "num_beams":1})
    return output["text"]
audio_paths = []
for record in dataset_head:
    path = Path(record["id"] + ".wav")
    sf.write(path, record["audio"]["array"], record["audio"]["sampling_rate"], subtype="PCM_16")
    audio_paths.append(path.resolve())
transcripts = [transcribe_speech(str(path)) for path in audio_paths]
for record, transcript in zip(dataset_head, transcripts):
    print(record["id"] + ":", transcript)

## Example 10

In [ ]:
mic_transcribe = gr.Interface(fn=transcribe_speech, inputs=gr.Audio(sources="microphone", type="filepath"), outputs=gr.Textbox(label="Transcription", lines=3), flagging_mode="never", api_name="transcribe_microphone", analytics_enabled=False)
print("Microphone transcription interface created.")

## Example 11

In [ ]:
file_transcribe = gr.Interface(fn=transcribe_speech, inputs=gr.Audio(sources="upload", type="filepath"), outputs=gr.Textbox(label="Transcription", lines=3), flagging_mode="never", api_name="transcribe_upload", analytics_enabled=False)
print("File-upload transcription interface created.")

## Example 12

In [ ]:
with demo:
    gr.TabbedInterface([mic_transcribe, file_transcribe], ["Transcribe Microphone", "Transcribe Audio File"])
SHARE_DEMO = False
_, local_url, share_url = demo.launch(share=SHARE_DEMO, server_name="127.0.0.1", server_port=int(os.environ.get("PORT1", "3009")), prevent_thread_lock=True, quiet=True)
print("Demo URL:", local_url)
print("Public sharing enabled:", SHARE_DEMO)
print("Public share URL:", share_url)

## Independent regression checks

In [ ]:
import re
import urllib.request
from gradio_client import Client, handle_file
expected_ids = [f"374-180298-{i:04d}" for i in range(5)]
assert [r["id"] for r in dataset_head] == expected_ids
assert all(r["audio"]["sampling_rate"] == 16000 for r in dataset_head)
assert all(np.isfinite(r["audio"]["array"]).all() for r in dataset_head)
assert len(transcripts) == 5 and all(t.strip() for t in transcripts) and not asr.model.training
def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())
def edit_distance(left, right):
    row = list(range(len(right)+1))
    for i,a in enumerate(left,1):
        current = [i]
        for j,b in enumerate(right,1):current.append(min(current[-1]+1,row[j]+1,row[j-1]+(a!=b)))
        row = current
    return row[-1]
errors, count = 0, 0
try:
    for record, transcript in zip(dataset_head, transcripts):
        reference, actual = words(record["text"]), words(transcript)
        distance = edit_distance(reference, actual)
        assert distance/len(reference) <= 0.2
        errors += distance; count += len(reference)
        print(record["id"], ": word edits", distance, "/", len(reference))
    assert errors/count <= 0.1
    with urllib.request.urlopen(local_url, timeout=30) as response:
        assert response.status == 200
    client = Client(local_url, verbose=False)
    for api_name in ["/transcribe_upload", "/transcribe_microphone"]:
        result = client.predict(handle_file(str(audio_paths[0])), api_name=api_name)
        assert result == transcripts[0]
        print(api_name, ": real uploaded-file request matches direct function output.")
    raw = np.asarray(dataset_head[0]["audio"]["array"], dtype=np.float32)
    direct = asr({"raw":raw,"sampling_rate":16000}, generate_kwargs={"do_sample":False,"num_beams":1})["text"]
    assert direct == transcripts[0]
    assert SHARE_DEMO is False and share_url is None
    assert all(p.grad is None for p in asr.model.parameters())
    print("Five full original clips verified; total word edits:", errors, "/", count, "| WER:", f"{errors/count:.6f}")
    print("Both local Gradio routes, full-file/array consistency and no-public-sharing checks passed. Microphone hardware was not recorded.")
finally:
    demo.close()
    print("Temporary local demo closed. Re-run the launch cell to interact with it.")

## Original source appendix

Full original blocks from raw source SHA-256 71bdfbf09f4e2410f7c7490146a009dbd964b8dc2ac08673858633e0b39ac2b6. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
!pip install transformers
!pip install -U datasets
!pip install soundfile
!pip install librosa
!pip install gradio
```

### Original block 2

```text
from datasets import load_dataset
dataset = load_dataset("librispeech_asr",
                       split="train.clean.100",
                       streaming=True,
                       trust_remote_code=True)
```

### Original block 3

```text
example = next(iter(dataset))
dataset_head = dataset.take(5)
list(dataset_head)
```

### Original block 4

```text
list(dataset_head)[2]
```

### Original block 5

```text
from IPython.display import Audio as IPythonAudio

IPythonAudio(example["audio"]["array"],
             rate=example["audio"]["sampling_rate"])
```

### Original block 6

```text
from transformers import pipeline
asr = pipeline(task="automatic-speech-recognition",
               model="./models/distil-whisper/distil-small.en")
```

### Original block 7

```text
asr.feature_extractor.sampling_rate
```

### Original block 8

```text
example['audio']['sampling_rate']
```

### Original block 9

```text
import os
import gradio as gr
demo = gr.Blocks()
```

### Original block 10

```text
def transcribe_speech(filepath):
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        return ""
    output = asr(filepath)
    return output["text"]
```

### Original block 11

```text
mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never")
```

### Original block 12

```text
file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never",
)
```

### Original block 13

```text
with demo:74
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True, 
            server_port=int(os.environ['PORT1']))
```

## Original source outputs

[{‘file’: ‘374–180298–0000.flac’, ‘audio’: {‘path’: ‘374–180298–0000.flac’, ‘array’: array([ 7.01904297e-04, 7.32421875e-04, 7.32421875e-04, …, -2.74658203e-04, -1.83105469e-04, -3.05175781e-05]), ‘sampling_rate’: 16000}, ‘text’: ‘CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED’, ‘speaker_id’: 374, ‘chapter_id’: 180298, ‘id’: ‘374–180298–0000’}, {‘file’: ‘374–180298–0001.flac’, ‘audio’: {‘path’: ‘374–180298–0001.flac’, ‘array’: array([-9.15527344e-05, -1.52587891e-04, -1.52587891e-04, …, -2.13623047e-04, -1.83105469e-04, -2.74658203e-04]), ‘sampling_rate’: 16000}, ‘text’: “MARGUERITE TO BE UNABLE TO LIVE APART FROM ME IT WAS THE DAY AFTER THE EVENING WHEN SHE CAME TO SEE ME THAT I SENT HER MANON LESCAUT FROM THAT TIME SEEING THAT I COULD NOT CHANGE MY MISTRESS’S LIFE I CHANGED MY OWN”, ‘speaker_id’: 374, ‘chapter_id’: 180298, ‘id’: ‘374–180298–0001’}, {‘file’: ‘374–180298–0002.flac’, ‘audio’: {‘path’: ‘374–180298–0002.flac’, ‘array’: array([-2.44140625e-04, -2.44140625e-04, -1.83105469e-04, …, 1.83105469e-04, 3.05175781e-05, -1.52587891e-04]), ‘sampling_rate’: 16000}, ‘text’: ‘I WISHED ABOVE ALL NOT TO LEAVE MYSELF TIME TO THINK OVER THE POSITION I HAD ACCEPTED FOR IN SPITE OF MYSELF IT WAS A GREAT DISTRESS TO ME THUS MY LIFE GENERALLY SO CALM’, ‘speaker_id’: 374, ‘chapter_id’: 180298, ‘id’: ‘374–180298–0002’}]

{‘file’: ‘374–180298–0002.flac’, ‘audio’: {‘path’: ‘374–180298–0002.flac’, ‘array’: array([-2.44140625e-04, -2.44140625e-04, -1.83105469e-04, …, 1.83105469e-04, 3.05175781e-05, -1.52587891e-04]), ‘sampling_rate’: 16000}, ‘text’: ‘I WISHED ABOVE ALL NOT TO LEAVE MYSELF TIME TO THINK OVER THE POSITION I HAD ACCEPTED FOR IN SPITE OF MYSELF IT WAS A GREAT DISTRESS TO ME THUS MY LIFE GENERALLY SO CALM’, ‘speaker_id’: 374, ‘chapter_id’: 180298, ‘id’: ‘374–180298–0002’}

16000

16000